In [1]:
import os
import pathlib
import sys
import numpy as np
import tensorflow_datasets as tfds
import tensorflow_text as text
from tensorflow_text.tools.wordpiece_vocab import bert_vocab_from_dataset as bert_vocab
import tensorflow as tf
from SubWord_Tokenizer import CustomTokenizer

# Dataset Loading and Creating Vocabulary

In [9]:
X_pten, metadata_pten = tfds.load('ted_hrlr_translate/pt_to_en', with_info = True, as_supervised = True)
X_ruen, metadata_ruen = tfds.load('ted_hrlr_translate/ru_to_en', with_info = True, as_supervised = True)
X_Train_pten, X_Validate_pten = X_pten["train"], X_pten["validation"]
X_Train_ruen, X_Validate_ruen = X_ruen["train"], X_ruen["validation"]

In [15]:
train_enpt = X_Train_pten.map(lambda pt, en: en)
train_pten = X_Train_pten.map(lambda pt, en: pt)
train_enru = X_Train_ruen.map(lambda ru, en: en)
train_ruen = X_Train_ruen.map(lambda ru, en: ru)

In [2]:
reserved_tokens = ["[START]", "[END]", "[PAD]", "[UNK]"]

In [17]:
bert_tokenizer_params = dict(lower_case = True)
bert_vocab_args = dict(vocab_size = 10000, reserved_tokens = reserved_tokens, bert_tokenizer_params = bert_tokenizer_params, learn_params = {})

In [18]:
enpt_vocab = bert_vocab.bert_vocab_from_dataset(train_enpt.batch(1000).prefetch(2), **bert_vocab_args)
pten_vocab = bert_vocab.bert_vocab_from_dataset(train_pten.batch(1000).prefetch(2), **bert_vocab_args)
enru_vocab = bert_vocab.bert_vocab_from_dataset(train_enru.batch(1000).prefetch(2), **bert_vocab_args)
ruen_vocab = bert_vocab.bert_vocab_from_dataset(train_ruen.batch(1000).prefetch(2), **bert_vocab_args)

In [21]:
def write_vocab_file(path, vocab):
    with open(path, "w", encoding="utf-8") as f:
        for token in vocab:
            print(token, file = f)

In [22]:
write_vocab_file("enpt_vocab.txt", enpt_vocab)
write_vocab_file("pten_vocab.txt", pten_vocab)
write_vocab_file("enru_vocab.txt", enru_vocab)
write_vocab_file("ruen_vocab.txt", ruen_vocab)

# Creating Tokenizer and Exporting

In [3]:
tokenizers_pten = tf.Module()
tokenizers_pten.en = CustomTokenizer(reserved_tokens, "enpt_vocab.txt")
tokenizers_pten.pt = CustomTokenizer(reserved_tokens, "pten_vocab.txt")
tokenizers_ruen = tf.Module()
tokenizers_ruen.en = CustomTokenizer(reserved_tokens, "enru_vocab.txt")
tokenizers_ruen.ru = CustomTokenizer(reserved_tokens, "ruen_vocab.txt")

In [4]:
model_name = 'pt_en_converter'
tf.saved_model.save(tokenizers_pten, model_name)
model_name = 'ru_en_converter'
tf.saved_model.save(tokenizers_ruen, model_name)

INFO:tensorflow:Assets written to: pt_en_converter\assets
INFO:tensorflow:Assets written to: ru_en_converter\assets
